# PostgreSQL 선택적 행 정리

리팩토링 전후의 실행 데이터가 섞이지 않도록, 허용된 테이블에서 **정확한 `run_uid` 범위만** 미리 확인하고 삭제합니다.

안전 원칙:

- 기본값은 DB 연결과 삭제가 모두 꺼져 있습니다.
- 임의 SQL, 임의 테이블명, 조건 없는 전체 테이블 삭제는 받지 않습니다.
- 공유 원본 테이블 `images`는 이 노트북에서 삭제하지 않습니다.
- 미리보기 뒤 행 수·DB·run 상태가 바뀌면 트랜잭션 전체를 롤백합니다.
- 완료된 로컬 run은 명시적으로 보호 해제하지 않으면 삭제할 수 없습니다.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from sqlalchemy.orm import Session


def find_project_root(start: Path) -> Path:
    resolved = start.resolve()
    for candidate in (resolved, *resolved.parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun 프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.database.cleanup import (
    SCOPE_EXACT_RUN_UID,
    SCOPE_LEGACY_NULL_RUN_UID,
    build_cleanup_plan,
    collect_run_inventory,
    collect_table_totals,
    execute_cleanup_plan,
    write_cleanup_audit,
)
from research.database.connection import (
    check_database_health,
    create_database_engine,
)
from research.database.settings import load_database_settings

print(f"PROJECT_ROOT={PROJECT_ROOT}")


## 1. 실행 설정

`TABLE_GROUPS_SELECTED`와 `TABLE_NAMES_SELECTED`를 합쳐 삭제 후보를 정합니다.

- `image_embeddings`: `embedding_512/448/384/256/128/pq`
- `template_embeddings`: `template_embedding_512/448/384/256/128`
- `research_run_children`: split/template/search/calibration 결과
- 개별 선택 예: `TABLE_GROUPS_SELECTED=[]`, `TABLE_NAMES_SELECTED=["embedding_256"]`
- `legacy_null_run_uid`는 과거 `run_uid IS NULL`인 image embedding 행에만 사용합니다.


In [ ]:
# 1단계: 인벤토리 확인 시에만 True로 바꿉니다.
CONNECT_TO_DATABASE = False

# 일반 정리는 exact_run_uid를 사용합니다.
SCOPE_KIND = SCOPE_EXACT_RUN_UID  # 또는 SCOPE_LEGACY_NULL_RUN_UID
RUN_UID = ""  # exact_run_uid일 때 삭제할 run_uid를 정확히 입력

# 그룹과 개별 테이블을 함께 지정할 수 있습니다.
TABLE_GROUPS_SELECTED = ["image_embeddings", "template_embeddings"]
TABLE_NAMES_SELECTED = []

# research_runs 부모 행까지 지울 때만 True. 모든 FK 자식이 자동 포함됩니다.
INCLUDE_RESEARCH_RUN_RECORD = False

# 완료된 run_manifest 또는 research_runs.status=completed 삭제 보호 해제.
ALLOW_COMPLETED_RUN_DELETE = False

# 2단계: 미리보기의 확인 문자열을 붙여 넣은 뒤 두 값을 함께 설정합니다.
EXECUTE_DELETE = False
CONFIRMATION_TOKEN = ""

# 실제 삭제가 성공하면 runs/database_cleanup 아래에 감사 JSON을 남깁니다.
WRITE_AUDIT_OUTPUT = True

{
    "connect": CONNECT_TO_DATABASE,
    "scope_kind": SCOPE_KIND,
    "run_uid": RUN_UID or None,
    "table_groups": TABLE_GROUPS_SELECTED,
    "table_names": TABLE_NAMES_SELECTED,
    "include_research_run_record": INCLUDE_RESEARCH_RUN_RECORD,
    "allow_completed_run_delete": ALLOW_COMPLETED_RUN_DELETE,
    "execute_delete": EXECUTE_DELETE,
}


## 2. DB 연결 및 읽기 전용 인벤토리

이 단계는 스키마를 생성·변경하지 않습니다. 비밀번호는 화면에 출력하지 않습니다.


In [ ]:
engine = None
database_health = None

if CONNECT_TO_DATABASE:
    database_settings = load_database_settings()
    display(pd.DataFrame([database_settings.redacted()]))
    engine = create_database_engine(database_settings)
    database_health = check_database_health(engine)
    if database_health["missing_tables"] or database_health["schema_issues"]:
        raise RuntimeError(
            "DB schema가 현재 모델과 다릅니다. 삭제하지 말고 먼저 health 결과를 확인하세요."
        )
    display(
        pd.DataFrame(
            [
                {
                    "database": database_health["database"],
                    "user": database_health["user"],
                    "vector_extension_version": database_health[
                        "vector_extension_version"
                    ],
                    "missing_tables": len(database_health["missing_tables"]),
                    "schema_issues": len(database_health["schema_issues"]),
                }
            ]
        )
    )
else:
    print("안전 기본값: DB에 연결하지 않았습니다. CONNECT_TO_DATABASE=True로 바꾸세요.")


In [ ]:
table_totals = []
run_inventory = []

if engine is not None:
    with Session(engine) as session:
        table_totals = [item.as_dict() for item in collect_table_totals(session)]
        run_inventory = [item.as_dict() for item in collect_run_inventory(session)]

    print("전체 관리 테이블 행 수")
    display(pd.DataFrame(table_totals))
    print("run_uid별 행 수")
    if run_inventory:
        display(
            pd.DataFrame(run_inventory).sort_values(
                ["run_uid", "table_name"], na_position="first"
            )
        )
    else:
        print("run_uid가 연결된 행이 없습니다.")
else:
    print("DB 연결이 꺼져 있어 인벤토리를 건너뜁니다.")


## 3. 삭제 계획 미리보기

여기까지는 `SELECT`만 수행합니다. 표의 행 수와 완료 run 보호 상태를 확인한 뒤, 실행 가능한 경우 출력되는 확인 문자열 전체를 상단 `CONFIRMATION_TOKEN`에 붙여 넣습니다.


In [ ]:
cleanup_plan = None
scope_is_configured = (
    SCOPE_KIND == SCOPE_LEGACY_NULL_RUN_UID
    or (SCOPE_KIND == SCOPE_EXACT_RUN_UID and bool(RUN_UID.strip()))
)

if engine is None:
    print("DB 연결이 꺼져 있어 미리보기를 건너뜁니다.")
elif not scope_is_configured:
    print("RUN_UID를 입력해야 exact_run_uid 미리보기를 생성할 수 있습니다.")
else:
    with Session(engine) as session:
        cleanup_plan = build_cleanup_plan(
            session,
            scope_kind=SCOPE_KIND,
            run_uid=RUN_UID or None,
            table_groups=TABLE_GROUPS_SELECTED,
            table_names=TABLE_NAMES_SELECTED,
            include_research_run_record=INCLUDE_RESEARCH_RUN_RECORD,
            allow_completed_run=ALLOW_COMPLETED_RUN_DELETE,
            project_root=PROJECT_ROOT,
        )

    display(pd.DataFrame([item.as_dict() for item in cleanup_plan.table_rows]))
    display(
        pd.DataFrame(
            [
                {
                    "database": cleanup_plan.database,
                    "database_user": cleanup_plan.database_user,
                    "scope_kind": cleanup_plan.scope_kind,
                    "run_uid": cleanup_plan.run_uid,
                    "total_rows": cleanup_plan.total_rows,
                    "research_run_status": cleanup_plan.research_run_status,
                    "plan_digest": cleanup_plan.plan_digest,
                    "executable": cleanup_plan.executable,
                }
            ]
        )
    )
    for warning in cleanup_plan.warnings:
        print(f"WARNING: {warning}")
    for blocker in cleanup_plan.blockers:
        print(f"BLOCKED: {blocker}")
    if cleanup_plan.confirmation_token:
        print("확인 문자열(공백 포함 그대로 복사):")
        print(cleanup_plan.confirmation_token)


## 4. 트랜잭션 DELETE

`EXECUTE_DELETE=True`만으로는 실행되지 않습니다. 같은 실행에서 다시 만든 계획과 `CONFIRMATION_TOKEN`이 정확히 일치해야 하며, PostgreSQL 테이블 잠금·재미리보기·행 수 검증을 모두 통과해야 커밋합니다.


In [ ]:
cleanup_report = None

if EXECUTE_DELETE:
    if engine is None:
        raise RuntimeError("EXECUTE_DELETE=True이지만 DB 연결이 없습니다.")
    if cleanup_plan is None:
        raise RuntimeError("실행 가능한 최신 미리보기가 없습니다.")
    if not CONFIRMATION_TOKEN:
        raise RuntimeError("미리보기의 CONFIRMATION_TOKEN을 정확히 입력하세요.")
    cleanup_report = execute_cleanup_plan(
        engine,
        cleanup_plan,
        confirmation_token=CONFIRMATION_TOKEN,
        project_root=PROJECT_ROOT,
    )
    display(pd.DataFrame([cleanup_report.as_dict()]))
else:
    print("안전 기본값: EXECUTE_DELETE=False이므로 DELETE를 실행하지 않았습니다.")


In [ ]:
audit_path = None
if cleanup_report is not None and WRITE_AUDIT_OUTPUT:
    audit_path = write_cleanup_audit(
        cleanup_report,
        PROJECT_ROOT / "runs" / "database_cleanup",
    )
    print(f"감사 기록: {audit_path}")
elif cleanup_report is not None:
    print("WARNING: 삭제는 완료됐지만 감사 JSON 기록은 꺼져 있습니다.")

if engine is not None:
    engine.dispose()


## 권장 사용 순서

1. `CONNECT_TO_DATABASE=True`, `EXECUTE_DELETE=False`로 위에서 아래까지 실행합니다.
2. 인벤토리에서 삭제할 `run_uid`와 테이블을 고르고 미리보기 행 수를 확인합니다.
3. 완료 run이면 정말 폐기할 실행인지 먼저 확인한 뒤에만 `ALLOW_COMPLETED_RUN_DELETE=True`로 바꿉니다.
4. 출력된 확인 문자열을 `CONFIRMATION_TOKEN`에 붙이고 `EXECUTE_DELETE=True`로 바꿉니다.
5. 커널을 재시작하고 다시 위에서 아래까지 실행합니다.
6. `runs/database_cleanup/`의 감사 JSON과 재조회된 인벤토리를 확인합니다.

`images`까지 제거하거나 조건 없는 전체 테이블 초기화가 필요하면 이 노트북을 우회하지 말고 별도의 스키마·FK 검토를 먼저 수행합니다.
